# Example use-case: LISA UCB convolution of double white-dwarf binaries at current-day

This example fleshes out the steps required to estimate the population of observable double white dwarf systems in the LISA band. Relevant studies are: https://arxiv.org/abs/2405.20484





Several ingredients are necessary here:
- population-synthesis results that contain white-dwarfs
- a Milky-Way galaxy star formation rate history model
- a method to evolve 

Convolution-by-sampling was developed especially for this project, as we want to 'generate' double white dwarf systems at a certain lookback time, and evolve them (through gravitational-wave radiation) to the present day.

The convolution broadly is done as follows:
- In a given lookback-time bin we calculate the total mass formed into stars
- We use that to generate double white dwarf systems (using mass_formed * yield-per-mass-formed) 
- We assign a birth time to these systems (with values bound by the edges of the lookback time bin)
- We 'evolve' these systems up to the current day under the influence of gravitational-wave radiation. We make use of Legwork ([Wagg et al 2021](https://ui.adsabs.harvard.edu/abs/2022ApJS..260...52W/abstract)) in this example.
- Filter out certain systems (those that would interact, those that would merge, etc)
- Calculate detection probabilities for the rest based on their position (either randomly assigned or motivated by a spatially-defined SFH) in the Milkyway and their system properties.
- Use this information to predict observable populations of DWD systems

In the following piece of code I show how we do this.

In [ ]:
# """
# Functions to convolve the T0 format with sampling

# TODO: move the calculations to the post-convolution hook
# TODO: determine which systems that are (at present day) in the lisa frequency range should have interacted through RLOF
# TODO: of the systems that are not RLOFing and are within the lisa waveband, store: indices, source.f_orb_now. the rest can be retrieved elsewhere
# """

# import os
# import json
# import time
# import copy
# import astropy.units as u
# import legwork as lw
# import numpy as np
# import astropy.constants as const
# import pandas as pd
# import h5py

# from syntheticstellarpopconvolve import convolve, default_convolution_config
# from syntheticstellarpopconvolve.general_functions import temp_dir

# # from mass_normalisation import get_mass_norm

# # from DrawPositionsSeparable import sample_distances_simple
# from syntheticstellarpopconvolve.convolve_stochastically import (
#     select_dict_entries_with_new_indices,
# )

# TMP_DIR = temp_dir("code", "convolve_stochastically", clean_path=True)

# import numpy as np
# from scipy.interpolate import CubicSpline

# #!/usr/bin/env python3
# # -*- coding: utf-8 -*-
# """
# Created on Wed Jun  5 18:05:24 2024

# @author: alexey
# """

# import numpy as np
# import pandas as pd
# import scipy as sp
# from scipy.optimize import root
# import sys

# sys.path.insert(1, "./PyModules/")

# import astropy.units as u
# import astropy.coordinates as coord

# # Units are kpc, Gyr


# def zCDFInv(Xiz, Hz):
#     zCoord = -Hz * np.log(1 - Xiz)
#     return zCoord


# def RCDFInv(Xir, Hr):
#     # Get the parameters for the inverse CDF
#     def RCD(R):
#         Res = (1 - np.exp(-R / Hr)) - (R / Hr) * np.exp(-R / Hr) - Xir
#         return Res

#     Sol = sp.optimize.root_scalar(RCD, bracket=(0.0001 * Hr, 20 * Hr))
#     if Sol.converged:
#         R = Sol.root
#     else:
#         print("The radial solution did not converge")
#         sys.exit()
#     return R


# def Sample1D(Hr, Hz):
#     RRand = np.random.uniform()
#     ZRand = np.random.uniform()
#     ZSign = np.sign(np.random.uniform() - 0.5)
#     R = RCDFInv(RRand, Hr)
#     Z = zCDFInv(ZRand, Hz) * ZSign
#     Th = 2.0 * np.pi * np.random.uniform()
#     X = R * np.cos(Th)
#     Y = R * np.sin(Th)
#     Age = np.random.uniform(0, 12)

#     XRel = X - 8.0
#     YRel = Y
#     ZRel = Z

#     RRel = np.sqrt(XRel**2 + YRel**2 + ZRel**2)

#     ResDict = {
#         "Age": Age,
#         "Xkpc": X,
#         "Ykpc": Y,
#         "Zkpc": Z,
#         "Rkpc": R,
#         "Th": Th,
#         "XRelkpc": XRel,
#         "YRelkpc": YRel,
#         "ZRelkpc": ZRel,
#         "RRelkpc": RRel,
#     }

#     return ResDict


# def Sample1DPop(NBin, Hr, Hz):
#     RRandSet = np.random.uniform(0, 1, NBin)
#     ZRandSet = np.random.uniform(0, 1, NBin)
#     ZSignSet = np.sign(np.random.uniform(0, 2, NBin) - 1)
#     RSet = np.asarray([RCDFInv(RRandSet[i], Hr) for i in range(NBin)], dtype=float)
#     ZSet = (
#         np.asarray([zCDFInv(ZRandSet[i], Hz) for i in range(NBin)], dtype=float)
#         * ZSignSet
#     )
#     ThSet = np.random.uniform(0, 2.0 * np.pi, NBin)
#     XSet = RSet * np.cos(ThSet)
#     YSet = RSet * np.sin(ThSet)
#     AgeSet = np.random.uniform(0, 12, NBin)
#     IDSet = np.arange(NBin) + 1

#     ResDict = {
#         "ID": IDSet,
#         "Ages": AgeSet,
#         "Xkpc": XSet,
#         "Ykpc": YSet,
#         "Zkpc": ZSet,
#         "Rkpc": RSet,
#         "Th": ThSet,
#     }
#     ResDF = pd.DataFrame(ResDict)
#     return ResDF


# ExportTable = False

# if ExportTable:
#     NBin = 10**4
#     Hr = 4
#     Hz = 0.5

#     Res = Sample1DPop(NBin, Hr, Hz)

#     Res.to_csv("./GalTest.csv", index=False)


# def sample_distances_simple(NBin):
#     """
#     Simple distance sampler using the
#     """

#     Hr = 4
#     Hz = 0.5

#     Res = Sample1DPop(NBin, Hr, Hz)

#     del Res["Ages"]
#     del Res["ID"]
#     del Res["Th"]
#     del Res["Rkpc"]

#     galcen_distance = 8.122  # * u*kpc

#     Res["Xkpc_rel"] = Res["Xkpc"] - galcen_distance
#     Res["Ykpc_rel"] = Res["Ykpc"]
#     Res["Zkpc_rel"] = Res["Zkpc"]

#     Res["Distance_to_sun"] = np.sqrt(
#         (Res["Xkpc_rel"] ** 2) + (Res["Ykpc_rel"] ** 2) + (Res["Zkpc_rel"])
#     )

#     return Res["Distance_to_sun"].to_numpy() * u.kpc

# def get_period(semimajor_axis, m1, m2):
#     """
#     function to get the periods of the systems
#     """

#     mtot = m1 + m2
#     p2 = ((semimajor_axis**3) * 4 * np.pi**2) / (const.G * mtot)
#     p = np.sqrt(p2)

#     return p.to(u.yr)

# def get_bin_frac_ratio(IC_model, binary_fraction=0.5):

#     # these are the hard coded ratios based on initial conditions sampling
#     # tests done by K. Breivik using binary fractions from 0.1-1.0
#     ratio_dict = {
#         "m2_min_05": [
#             5.497065159607499,
#             4.028385607219321,
#             3.1320443249502383,
#             2.541669993010094,
#             2.105948317768266,
#             1.786314805728416,
#             1.5164506548687628,
#             1.313816074228385,
#             1.141178996587137,
#             0.9968331286953663,
#             0.8730241509546721,
#             0.7696547388705937,
#             0.6788737987994921,
#             0.5998311488995663,
#             0.5308139080430776,
#             0.4699942084336498,
#             0.41210469109378484,
#             0.36061186819185237,
#             0.31590620434046357,
#             0.27425387234835646,
#             0.23543985197379194,
#             0.19987998623485503,
#             0.16926018076106297,
#             0.13969418254623414,
#             0.11143821417288673,
#             0.08563401307212755,
#             0.06239655229658728,
#             0.03973086411370632,
#             0.019317445609880482,
#             0.0,
#         ],
#         "log_uniform_porb": [
#             5.886862859310299,
#             4.3420507204072525,
#             3.405093627693288,
#             2.7466102648326007,
#             2.27052928301355,
#             1.9273925451530112,
#             1.6405699036728623,
#             1.4078089101945124,
#             1.2331941437437164,
#             1.069552690381338,
#             0.9456643329362575,
#             0.8327939389648276,
#             0.7359840090556246,
#             0.6480801205088771,
#             0.5722387172813348,
#             0.5039632125801372,
#             0.4440067853899282,
#             0.3913613364579611,
#             0.3402636518883438,
#             0.29843507961701765,
#             0.2543196815715736,
#             0.21741447251907284,
#             0.18304208420964435,
#             0.15107685764473266,
#             0.12041807829958329,
#             0.09269327057199453,
#             0.06744511227698262,
#             0.04274378038546733,
#             0.02137711323393566,
#             0.0,
#         ],
#         "ecc_uniform": [
#             5.889211849314321,
#             4.361889407239361,
#             3.4020828542097057,
#             2.746002579297502,
#             2.2673692464054183,
#             1.9141940732871088,
#             1.6405959517007267,
#             1.4094329827706373,
#             1.230470758764182,
#             1.072042568070819,
#             0.943431196191181,
#             0.8333833246264916,
#             0.7309333535548196,
#             0.6475817013002746,
#             0.574417702589979,
#             0.5056945660498336,
#             0.44488117789149545,
#             0.3884654642073608,
#             0.3388900220256573,
#             0.2962626999212119,
#             0.2557025971124346,
#             0.21685490011138606,
#             0.18305848181583317,
#             0.15006625511716296,
#             0.12090585737779058,
#             0.09221860126908878,
#             0.06728132687768605,
#             0.043606002397485674,
#             0.021239106748386388,
#             0.0,
#         ],
#         "qmin_01": [
#             5.823149995795932,
#             4.257871067861341,
#             3.3350623602831586,
#             2.701648948025014,
#             2.241763868198779,
#             1.8890779951715042,
#             1.6099148365641083,
#             1.3953172844589194,
#             1.2110201528032853,
#             1.0592751609913709,
#             0.9225325052920952,
#             0.8191978467059325,
#             0.7180939238557436,
#             0.6326983982513528,
#             0.5640129919280746,
#             0.49581103805511545,
#             0.43644888054132813,
#             0.3834832365660745,
#             0.3356910316544453,
#             0.29248525094604144,
#             0.2511267079306599,
#             0.21240840773299557,
#             0.1776591271238454,
#             0.14765505334148785,
#             0.11783814018942763,
#             0.09097678374483624,
#             0.06650626075146698,
#             0.04210676407248778,
#             0.0203586397778618,
#             0.0,
#         ],
#         "fiducial": [
#             5.932296012899532,
#             4.357823767133379,
#             3.394145723221763,
#             2.743531290685849,
#             2.261347038645783,
#             1.9173522895783706,
#             1.6465752814658685,
#             1.409849902360307,
#             1.232375306106761,
#             1.0745464651773011,
#             0.9431907171044254,
#             0.8331178361129032,
#             0.7317070852105148,
#             0.6506914750851542,
#             0.5707015794152772,
#             0.5048166662383198,
#             0.44379738107168043,
#             0.3913719830983486,
#             0.33791478738462305,
#             0.29476240868778614,
#             0.2551125640746029,
#             0.21752011832877077,
#             0.18147403979103155,
#             0.14971671211685017,
#             0.1209730374015313,
#             0.09372046390290077,
#             0.06743095018359142,
#             0.043359831467175966,
#             0.02054866077700454,
#             0.0,
#         ],
#         "ecc_thermal": [
#             5.9200035744176125,
#             4.341656072119606,
#             3.4004867453530436,
#             2.747854677340573,
#             2.2726296004773694,
#             1.9140533603622882,
#             1.6388693872876943,
#             1.4154780593615193,
#             1.2294748612513091,
#             1.0760766054593665,
#             0.9442728392343459,
#             0.8321788961090938,
#             0.7330076399499459,
#             0.6473674909796342,
#             0.5733259021802497,
#             0.5031744591982074,
#             0.44384269602503534,
#             0.38875209683437956,
#             0.3429980155840034,
#             0.2955812055263655,
#             0.25491188680860966,
#             0.21846399667917327,
#             0.18130416084339843,
#             0.14918121603752904,
#             0.12215748474240215,
#             0.09270863580218562,
#             0.06716028032728252,
#             0.04318939158189473,
#             0.020791845074920087,
#             0.0,
#         ],
#     }
#     binfracs = np.linspace(0.1, 1.0, 30)

#     # select the list of ratios based on the initial conditions model
#     ratio = ratio_dict[IC_model]

#     # set up a spline to get the ratio for any binfrac
#     r_spline = CubicSpline(binfracs, ratio)

#     return r_spline(binary_fraction)


# def get_mass_norm(IC_model, binary_fraction=0.5):
#     """selects the mass normalization for the
#     initial conditions sample set based on
#     the IC_model name and a binary fraction

#     Parameters
#     ----------
#     IC_model : `str`
#         initial conditions model chosen from:
#             ecc_uniform, ecc_thermal, porb_log_uniform, m2_min_05, qmin_01, fiducial

#     Returns
#     -------
#     mass_norm : `float`
#         the total ZAMS mass of the initial stellar population
#         including single and binary stars
#     """This example fleshes out the steps required to estimate the population of observable double white dwarf systems in the LISA band. Relevant studies are: https://arxiv.org/abs/2405.20484

# Several ingredients are necessary here:

#     population-synthesis results that contain white-dwarfs
#     a Milky-Way galaxy star formation rate history model
#     a method to evolve


#     mass_binaries = {
#         "ecc_uniform": 2720671.1164002735,
#         "ecc_thermal": 2700943.07050043,
#         "porb_log_uniform": 2713046.6197530716,
#         "m2_min_05": 2905718.830512573,
#         "qmin_01": 5510313.245766795,
#         "fiducial": 2697557.2681495477,
#     }

#     # get the ratio of singles to binaries for the selected binary fraction
#     ratio = get_bin_frac_ratio(IC_model, binary_fraction=binary_fraction)
#     mass_total = mass_binaries[IC_model] * (1 + ratio)

#     return mass_total



# def post_convolution_function(
#     config, job_dict, sfr_dict, data_dict, result_dict, convolution_instruction
# ):
#     """
#     Post-convolution function to handle integrating the systems forward in time and finding those that end up in the LISA waveband.

#     using local_indices to select everything and using Alexey's distance sampler to handle sampling the distances
#     """

#     # unpack data
#     system_indices = result_dict["indices"]
#     event_lookback_times = result_dict["event_lookback_times"]
#     local_indices = np.arange(len(system_indices))

#     # select system properties
#     sma = data_dict["semimajor_axis"][system_indices]
#     m_1 = data_dict["mass1"][system_indices]
#     m_2 = data_dict["mass2"][system_indices]
#     eccentricity = data_dict["eccentricity"][system_indices]
#     periods = get_period(sma, m_1, m_2)
#     f_orb_i = (1 / periods).to(u.Hz)

#     # sample distances
#     dist = sample_distances_simple(NBin=len(system_indices), age=age)

#     #########
#     # Set up legwork sources
#     sources = lw.source.Source(
#         m_1=m_1,
#         m_2=m_2,
#         ecc=eccentricity,
#         f_orb=f_orb_i,
#         dist=dist,
#         interpolate_g=len(local_indices) > 1000,
#     )

#     ##########
#     # TODO: use the below steps to improve the selection of systems in band
#     # ask; how long until separation = RLOF separatin
#     # then: how long does it take until with current separation to go to that seperation
#     # then compare that time to
#     # check if the distance matches a frequency thats within the waveband.

#     #########
#     # Evolve the systems until today
#     t_evol = event_lookback_times

#     sources.evolve_sources(t_evol)

#     f_orb_now = sources.f_orb

#     ####
#     # categorisations
#     lower_bound_LISA_passband = 1e-5 * u.Hz
#     upper_bound_LISA_passband = 1e-1 * u.Hz

#     # 1) doesnt enter lisa waveband today. so also nt in the past (maybe near future)
#     # 2) are currently in lisa band. maybe also in the past (but fro which point)
#     # 3) are merged now. but they ahve been in lisa band in the past (and from which point)
#     # 4) for both 2 and 3, we should filter out the 'interacting' systems

#     ##############
#     # determine (un)merged systems
#     local_indices_merged_systems = local_indices[f_orb_now >= 1e2 * u.Hz]
#     local_indices_unmerged_systems = local_indices[f_orb_now < 1e2 * u.Hz]
#     config["logger"].warning(
#         f"Of the total of {len(local_indices)} systems {len(local_indices_merged_systems)} are merged by today and {len(local_indices_unmerged_systems)} are not"
#     )

#     f_orb_now_unmerged_systems = f_orb_now[f_orb_now < 1e2 * u.Hz]

#     ##############
#     # determine unmerged systems in LISA passband
#     query_unmerged_systems_within_LISA_passband = (
#         f_orb_now_unmerged_systems >= lower_bound_LISA_passband
#     ) & (f_orb_now_unmerged_systems <= upper_bound_LISA_passband)

#     #
#     local_indices_unmerged_systems_within_LISA_passband = (
#         local_indices_unmerged_systems[query_unmerged_systems_within_LISA_passband]
#     )

#     print(
#         f"Of the {len(local_indices_unmerged_systems)} unmerged systems {len(local_indices_unmerged_systems_within_LISA_passband)} are within the lisa frequency passband ([{lower_bound_LISA_passband},{upper_bound_LISA_passband}])"
#     )

#     # return only data from now unmerged systems within the lisa passband

#     result_dict = select_dict_entries_with_new_indices(
#         sampled_data_dict=result_dict,
#         new_indices=local_indices_unmerged_systems_within_LISA_passband,
#     )

#     # add dists
#     result_dict["dists"] = (
#         dist[local_indices_unmerged_systems_within_LISA_passband] * u.kpc
#     )

#     return result_dict


# LIGHTWEIGHT = False

# ###################
# # Read T0 output
# start = time.time()

# if LIGHTWEIGHT:
#     BinCodex_events_filename = (
#         "/home/david/Desktop/bincodex_results/example_BinCodex.h5"
#     )
# else:
#     BinCodex_events_filename = "/home/david/Desktop/bincodex_results/Seba_BinCodex.h5"

# #
# BinCodex_T0_events = pd.read_hdf(
#     BinCodex_events_filename,
#     "T0",
# )

# ##################
# # update T0 output

# # get mass normalisation
# mass_normalisation_fiducial = get_mass_norm(IC_model="fiducial", binary_fraction=0.5)

# # set normalised yield
# BinCodex_T0_events["normalized_yield"] = 1 / mass_normalisation_fiducial

# # Query the dataset to select the formation of the WDs

# # to check if things start with some number its easier to turn them into strings
# BinCodex_T0_events["str_event"] = BinCodex_T0_events["event"].astype(str)
# BinCodex_T0_events["str_type1"] = BinCodex_T0_events["type1"].astype(str)
# BinCodex_T0_events["str_type2"] = BinCodex_T0_events["type2"].astype(str)

# # first, lets query the type-changing events. Any type-change will do
# wd_binaries = BinCodex_T0_events.query("str_event.str.startswith('1')")

# # The type should change to a WD-type (and the other should already be one)
# wd_binaries = wd_binaries.query("str_type1.str.startswith('2')")
# wd_binaries = wd_binaries.query("str_type2.str.startswith('2')")

# # to be sure lets only select the first ones that remain for each system
# # Drop duplicates based on 'system_id', keeping only the first occurrence
# # wd_binaries = wd_binaries.drop_duplicates(subset='UID', keep='first')

# # lets delete the string versions of the columns again
# wd_binaries = wd_binaries.drop(columns=["str_event", "str_type1", "str_type2"])

# # lets also delete the original dataframe
# del BinCodex_T0_events

# stop = time.time()
# print("created queried dataframe")
# print("took {}".format(stop - start))

# ##################
# #

# # create file
# input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
# output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
# input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# # Create groups main
# input_hdf5_file.create_group("input_data")
# input_hdf5_file.create_group("config")

# # add group for events
# input_hdf5_file.create_group("input_data/events")

# # Write population config to file
# input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# # close
# input_hdf5_file.close()

# # store the data frame in the hdf5file
# wd_binaries.to_hdf(input_hdf5_filename, key="input_data/events/stochastic_example")

# #
# convolution_config = copy.copy(default_convolution_config)
# convolution_config["input_filename"] = input_hdf5_filename
# convolution_config["output_filename"] = output_hdf5_filename
# convolution_config["tmp_dir"] = TMP_DIR
# convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
#     TMP_DIR, "interpolator_dict.p"
# )
# convolution_config["multiply_by_time_binsize"] = False

# ###
# # convolution instructions
# convolution_config["convolution_instructions"] = [
#     {
#         "input_data_type": "event",
#         "convolution_type": "sample",
#         "input_data_name": "stochastic_example",
#         "output_data_name": "stochastic_example",
#         "ignore_metallicity": True,
#         "post_convolution_function": post_convolution_function,
#         "data_column_dict": {
#             # required
#             "normalized_yield": "normalized_yield",
#             "delay_time": {"column_name": "time", "unit": u.Myr},
#         },
#     },
# ]

# #
# convolution_config["time_type"] = "lookback_time"
# # convolution_config["convolution_lookback_time_bin_edges"] = np.arange(0, 4, 0.5) * u.Gyr

# # construct the sfr-dict (NOTE: this uses absolute SFR, not metallicity dependent)
# sfr_dict = {}
# if LIGHTWEIGHT:
#     sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 1) * u.Gyr).to(u.yr)
# else:
#     sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 0.25) * u.Gyr).to(u.yr)

# sfr_dict["starformation_rate_array"] = (
#     1e-6 * np.ones(sfr_dict["lookback_time_bin_edges"].shape[0] - 1) * u.Msun / u.yr
# )  # example of a constant star-formation rate. this could be anything of course.

# # store
# convolution_config["SFR_info"] = sfr_dict

# input_hdf5_file = h5py.File(input_hdf5_filename, "r")

# # convolve
# convolve(config=convolution_config)

# print("finished convolution")


# # read out content and integrate until today
# with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:
#     print(
#         output_hdf5_file[
#             "output_data/event/stochastic_example/stochastic_example/convolution_results"
#         ].keys()
#     )

#     formation_time_bin_keys = list(
#         output_hdf5_file[
#             "output_data/event/stochastic_example/stochastic_example/convolution_results"
#         ].keys()
#     )

#     ################
#     #
#     total_in_waveband_lisa = 0

#     # loop over the formation-time bins
#     formation_time_bin_keys = sorted(
#         formation_time_bin_keys, key=lambda x: float(x.split(" ")[0])
#     )
#     for formation_time_bin_key in formation_time_bin_keys:

#         # formation_time_bin_key = "3500000000.0 yr"
#         print("=================================")
#         print(f"formation_time_bin_key: {formation_time_bin_key}")
#         print("=================================")

#         ###########
#         # Read out data

#         # convert units
#         unit_dict = json.loads(
#             output_hdf5_file[
#                 f"output_data/event/stochastic_example/stochastic_example/convolution_results/{formation_time_bin_key}"
#             ].attrs["units"]
#         )
#         unit_dict = {key: u.Unit(val) for key, val in unit_dict.items()}
#         print(unit_dict)

This example can be made more sophisticated by e.g.:
- Using a spatially-defined star-formation rate history. One can provide a list of starformation histories to the code, each element then representing a part of the grid where the SFR is defined in.
- Splitting 